# OpenBind-HIPPO

- **Target: D68EV3C**
- **Cycle: 01**

## Imports

In [2]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import pandas as pd
import plotly.express as px

## Config

In [3]:
target_name = "D68EV3C"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [4]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

 Creating HIPPO animal

name = D68EV3C

db_path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite

DEBUG: hippo.Database.__init__()

DEBUG: Database.path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite

DEBUG: hippo.Database.connect()

DEBUG: sqlite3.version='2.6.0'

 Success  Database connected @ /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/D68EV3C/D68EV3C.sqlite!

 Success  Initialised animal HIPPO("D68EV3C")!

## Queue BulkDock (re)placements

In [13]:
scaffolds = animal.compounds(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
elab_poses = scaffolds.elabs.poses
scaffolds, elab_poses

(compounds tagged openbind_d68ev3c_c1_scaffolds_chemok_bbok: {C × 77},
 {P × 231})

In [20]:
dedupe = set()
for pose in mrich.track(elab_poses):
    compound = pose.compound
    scaffolds = compound.scaffolds
    if len(scaffolds) != 1:
        mrich.warning("multiple", pose)
    scaffold = scaffolds[0]
    
    reference = pose.reference
    inspirations = pose.inspirations

    dedupe.add((scaffold.id, reference.id, tuple(inspirations.ids)))

    # elabs = scaffold.elabs

data = []

for scaffold_id, reference_id, inspiration_ids in dedupe:
    print(scaffold_id, reference_id, inspiration_ids)

    scaffold = animal.compounds[scaffold_id]
    reference = animal.poses[reference_id]
    inspirations = animal.poses[inspiration_ids]

    inspiration_d = dict()
    for i, name in enumerate(inspirations.names):
        inspiration_d[f"hit{i+1}"] = name
    
    for elab in mrich.track(scaffold.elabs):
        d = dict(smiles=elab.smiles)
        d.update(inspiration_d)
        data.append(d)

df = pd.DataFrame(data)
df.head()

Output()

33797 97
(44, 59)

Output()

30265 55
(106, 124)

Output()

46521 99
(98, 111)

Output()

,smiles,hit1,hit2
0,Cc1nn(CCNC(=O)CCc2c(C)[nH]c3ccccc23)cc1Cl,7gp2-a,7goy-a
1,Cc1[nH]c2ccccc2c1CCC(=O)NCCn1ncc(Cl)c1C,7gp2-a,7goy-a
2,Cc1c(Cl)cnn1CCNC(=O)CCc1cn(C)c2ccccc12,7gp2-a,7goy-a
3,Cc1nn(CCNC(=O)CCc2cn(C)c3ccccc23)cc1Cl,7gp2-a,7goy-a
4,Cc1ccc2[nH]cc(CCC(=O)NCCn3cc(Cl)c(C)n3)c2c1,7gp2-a,7goy-a


In [21]:
print(len(df))
df = df.drop_duplicates()
print(len(df))

1170

1170

In [22]:
df.to_csv("cycle_01/syndirella/elabs/d68ev3c_c1_elab_bulkdock_input.csv", index=False)
df

,smiles,hit1,hit2
0,Cc1nn(CCNC(=O)CCc2c(C)[nH]c3ccccc23)cc1Cl,7gp2-a,7goy-a
1,Cc1[nH]c2ccccc2c1CCC(=O)NCCn1ncc(Cl)c1C,7gp2-a,7goy-a
2,Cc1c(Cl)cnn1CCNC(=O)CCc1cn(C)c2ccccc12,7gp2-a,7goy-a
3,Cc1nn(CCNC(=O)CCc2cn(C)c3ccccc23)cc1Cl,7gp2-a,7goy-a
4,Cc1ccc2[nH]cc(CCC(=O)NCCn3cc(Cl)c(C)n3)c2c1,7gp2-a,7goy-a
...,...,...,...
1165,O=C(NC(Cc1c[nH]c2ccccc12)C(=O)N1CCC(n2cc(Cl)cn...,7gp2-a,7goy-a
1166,CCC(CNC(=O)C(Cc1c[nH]c2ccccc12)NC(=O)c1ccc(F)c...,7gp2-a,7goy-a
1167,Cc1nn(CCNC(=O)C(Cc2c[nH]c3ccccc23)NC(=O)c2ccc(...,7gp2-a,7goy-a
1168,COC(=O)C(Cc1cc[nH]n1)NC(=O)C1Cc2[nH]ncc2C(=O)N1,7go4-a,7gpo-a


In [11]:
# scaffold_poses = animal.poses(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
# elaborated_scaffolds = scaffold_poses.compounds.elabs.scaffolds
# scaffold_poses.compounds.elabs
# elaborated_scaffold_poses = animal.poses[set(elaborated_scaffolds.poses.ids).intersection(set(scaffold_poses.ids))]
# elaborated_scaffold_poses

## Define chemspace

In [5]:
scaffolds = animal.compounds(tag="openbind_d68ev3c_c1_scaffolds_chemok_bbok")
scaffolds

compounds tagged openbind_d68ev3c_c1_scaffolds_chemok_bbok: {C × 77}

In [6]:
elabs = scaffolds.elabs
elabs

{C × 1211}

In [7]:
posed_elabs = elabs.poses.compounds
posed_elabs

{C × 1170}

In [8]:
posed_elabs.add_tag("openbind_d68ev3c_c1_elabs")

Tagged {C × 1170} w/ "openbind_d68ev3c_c1_elabs"

In [9]:
posed_elabs.scaffolds.add_tag("openbind_d68ev3c_c1_elaborated_scaffolds")

Tagged {C × 3} w/ "openbind_d68ev3c_c1_elaborated_scaffolds"

In [10]:
chemspace = posed_elabs + posed_elabs.scaffolds
chemspace

{C × 1173}

In [11]:
full_recipe = hippo.Recipe.from_compounds(chemspace)

#compounds = 1173

Output()

Solving recipe combinations...

Output()

DEBUG: Calculating prices...

Picking cheapest from 1 options

In [ ]:
full_recipe.write_json("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace.json")

In [ ]:
full_recipe.write_CAR_csv("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace.csv")

In [ ]:
full_recipe.write_reactant_csv("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace_reactants.csv")

In [4]:
animal.tags.summary();

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ tag                                       ┃ num_compounds ┃ num_poses ┃ num_posed_compounds ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ BAD count                                 │ 0             │ 202       │ 156                 │
│ BulkDock Fragalysis export                │ 0             │ 351       │ 351                 │
│ GOOD count                                │ 0             │ 202       │ 156                 │
│ MEDIOCRE count                            │ 0             │ 202       │ 156                 │
│ Main status                               │ 0             │ 202       │ 156                 │
│ [Other] iter1_frags                       │ 0             │ 42        │ 42                  │
│ [Other] upload_1 2025-02-11               │ 0             │ 202       │ 156                 │
│ d68ev3c_c1_elab_bulkdock_input            │ 1170          │ 2340      │ 1170                │
│ fragmenstein_placed                       │ 0             │ 63108     │ 42916               │
│ hits                                      │ 156           │ 202       │ 156                 │
│ openbind_d68ev3c_c1_elaborated_scaffolds  │ 3             │ 0         │ 0                   │
│ openbind_d68ev3c_c1_elabs                 │ 157           │ 0         │ 0                   │
│ openbind_d68ev3c_c1_fragmenstein          │ 1665          │ 2003      │ 1665                │
│ openbind_d68ev3c_c1_knitwork_impure       │ 18230         │ 20662     │ 15107               │
│ openbind_d68ev3c_c1_knitwork_pure         │ 25521         │ 38103     │ 25473               │
│ openbind_d68ev3c_c1_scaffolds             │ 345           │ 345       │ 345                 │
│ openbind_d68ev3c_c1_scaffolds_chemok      │ 175           │ 0         │ 0                   │
│ openbind_d68ev3c_c1_scaffolds_chemok_bbok │ 77            │ 77        │ 77                  │
└───────────────────────────────────────────┴───────────────┴───────────┴─────────────────────┘

In [11]:
animal.scaffolds.elabs.poses.compounds

{C × 1170}

In [12]:
animal.elabs.poses.compounds.scaffolds

{C × 3}

## Export Chemspace

In [13]:
filtered_elab_poses = elabs.poses.filter(key="distance_score", value="2.0", operator="<=").filter(key="energy_score", value="0.0", operator="<=")
filtered_elab_poses.add_tag("openbind_d68ev3c_c1_elabs_filtered")
filtered_posed_elabs = filtered_elab_poses.compounds
filtered_posed_elabs.add_tag("openbind_d68ev3c_c1_elabs_filtered")
chemspace_filtered = filtered_posed_elabs + posed_elabs.scaffolds
chemspace_filtered

Tagged {P × 423} w/ "openbind_d68ev3c_c1_elabs_filtered"

Tagged {C × 216} w/ "openbind_d68ev3c_c1_elabs_filtered"

{C × 219}

In [14]:
chemspace_filtered_poses = filtered_elab_poses.get_best_placed_poses_per_compound() + posed_elabs.scaffolds.best_placed_poses
chemspace_filtered_poses

{P × 219}

In [15]:
%%time
chemspace_filtered_poses.to_fragalysis("cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace_filtered.sdf",
    method="openbind_d68ev3c_c1_elab_chemspace_filtered", 
    submitter_name="Max Winokan", 
    submitter_institution="DLS", 
    submitter_email="max.winokan@diamond.ac.uk",
    copy_reference_pdbs=True,
    metadata=False,
    tags=False,
    subsites=False,
)

DEBUG: 219 poses in set

DEBUG: 219 remaining after skipping null reference

DEBUG: 219 remaining after skipping null inspirations

#poses = 219

DEBUG: querying...

...

 DISK  Writing openbind_d68ev3c_c1_elab_chemspace_filtered_refs.zip...

[11:43:25] Molecule does not have explicit Hs. Consider calling AddHs()


 DISK  Writing 
/opt/xchem-fragalysis-2/maxwin/openbind-hippo/d68ev3c/cycle_01/syndirella/elabs/openbind_d68ev3c_c1_elab_chemspace_
filtered.sdf...

CPU times: user 399 ms, sys: 352 ms, total: 751 ms
Wall time: 6.91 s


,HIPPO Pose ID,smiles,inchikey,alias,HIPPO Compound ID,ROMol,energy_score,distance_score,inspiration_score,name,ref_mols,ref_pdb,_Name
0,6018,O=C(CCc1c[nH]c2ccccc12)NCCn1cc(Cl)cn1,IGFSVBYQUJOBES-UHFFFAOYSA-N,None,33797,<rdkit.Chem.rdchem.Mol object at 0x7f9f48da0270>,492.245977,9.278592,None,P6018,"7go7-a,B0426a",7go4-b,P6018
1,24064,N[C@@H](Cc1cc[nH]n1)C(=O)NCCc1cccc2cc[nH]c12,WGWCVZXHQCSGIO-UHFFFAOYSA-N,None,30265,<rdkit.Chem.rdchem.Mol object at 0x7f9f486bfec0>,1055.409110,6.189318,None,P24064,"7gok-a,7gqj-a",7gq5-b,P24064
2,42759,c1cn(-c2csc(CNc3ccc4nscc4c3)c2)cn1,WOHHKMXIYBOTTO-UHFFFAOYSA-N,None,46521,<rdkit.Chem.rdchem.Mol object at 0x7f9f486bfbf0>,-45.880960,2.458740,None,P42759,"7goj-a,7goa-b",7gqa-a,P42759
3,60973,COC(=O)[C@@H](Cc1cc[nH]n1)NC(=O)[C@@H]1Cc2[nH]...,AOVBBXNPKSDHLC-WDEREUQCSA-N,None,98112,<rdkit.Chem.rdchem.Mol object at 0x7f9f486bf100>,-28.865323,1.747510,None,P60973,"7go4-a,7gpo-a",7got-a,P60973
4,60976,Cc1nn(CCNC(=O)CCc2c[nH]c3cc(Cl)ccc23)cc1Cl,SDDKWSLVKYDDLW-UHFFFAOYSA-N,None,98218,<rdkit.Chem.rdchem.Mol object at 0x7f9f486bf290>,-0.682157,1.905970,None,P60976,"7goy-a,7gp2-a",7go7-a,P60976
...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,63439,Cc1c(Cl)c(C(F)(F)F)nn1CC(=O)NNC(=O)CC(O)(c1c[n...,CNYZEMKPXABBAJ-UHFFFAOYSA-N,CNYZEMKPXABBAJ-UHFFFAOYSA-N-P59-P44-P59-607688,99327,<rdkit.Chem.rdchem.Mol object at 0x7f9f486fbc40>,-14.257584,1.307028,None,CNYZEMKPXABBAJ-UHFFFAOYSA-N-P59-P44-P59-607688,"7goy-a,7gp2-a",7gp2-a,CNYZEMKPXABBAJ-UHFFFAOYSA-N-P59-P44-P59-607688
215,63458,Cc1c(Cl)c(C(F)(F)F)nn1C(C)C(=O)NNC(=O)CC(c1c[n...,ZLZDJYFLLJIGSW-UHFFFAOYSA-N,ZLZDJYFLLJIGSW-UHFFFAOYSA-N-P44-P44-P59-607688,99337,<rdkit.Chem.rdchem.Mol object at 0x7f9f486fb740>,-17.721631,1.344326,None,ZLZDJYFLLJIGSW-UHFFFAOYSA-N-P44-P44-P59-607688,"7goy-a,7gp2-a",7gp2-a,ZLZDJYFLLJIGSW-UHFFFAOYSA-N-P44-P44-P59-607688
216,63489,Cc1nn(CC(=O)NNC(=O)C(Cc2c[nH]c3ccccc23)NS(=O)(...,RXJAPEMQJJZPKV-UHFFFAOYSA-N,RXJAPEMQJJZPKV-UHFFFAOYSA-N-P59-P44-P59-607688,99352,<rdkit.Chem.rdchem.Mol object at 0x7f9f486fb7e0>,-21.607102,1.926282,None,RXJAPEMQJJZPKV-UHFFFAOYSA-N-P59-P44-P59-607688,"7goy-a,7gp2-a",7gp2-a,RXJAPEMQJJZPKV-UHFFFAOYSA-N-P59-P44-P59-607688
217,63490,CC(C)C(CC(=O)NNC(=O)Cn1nc(C(F)(F)F)c(Cl)c1C1CC...,LIXZRZWXPINAKN-UHFFFAOYSA-N,LIXZRZWXPINAKN-UHFFFAOYSA-N-P44-P44-P59-607688,99353,<rdkit.Chem.rdchem.Mol object at 0x7f9f486fb8d0>,-9.119938,1.859906,None,LIXZRZWXPINAKN-UHFFFAOYSA-N-P44-P44-P59-607688,"7goy-a,7gp2-a",7gp2-a,LIXZRZWXPINAKN-UHFFFAOYSA-N-P44-P44-P59-607688
